In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

In [2]:
# ============================================
# CONFIGURATION
# ============================================

BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 30

INPUT_SIZE = 500
LATENT_SIZE = 64

In [4]:
# ============================================
# LOAD DATA
# ============================================

X_train = np.load("Data/processed/X_train.npy")
X_dev = np.load("Data/processed/X_dev.npy")
X_test = np.load("Data/processed/X_test.npy")

print("Training data:", X_train.shape)
print("Development data:", X_dev.shape)
print("Test data:", X_test.shape)

Training data: (5985, 500)
Development data: (997, 500)
Test data: (998, 500)


In [5]:
# ============================================
# CONVERT TO PYTORCH TENSORS
# ============================================

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_dev_tensor = torch.tensor(X_dev, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

In [6]:
# ============================================
# DATA LOADERS
# ============================================

train_dataset = TensorDataset(X_train_tensor)
dev_dataset = TensorDataset(X_dev_tensor)
test_dataset = TensorDataset(X_test_tensor)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [7]:
# ============================================
# VANILLA AUTOENCODER
# ============================================

class Autoencoder(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder: 500 -> 256 -> 192 -> 128 -> 64
        self.encoder = nn.Sequential(
            nn.Linear(500, 256),
            nn.Tanh(),

            nn.Linear(256, 192),
            nn.Tanh(),

            nn.Linear(192, 128),
            nn.Tanh(),

            nn.Linear(128, 64)
        )

        # Decoder: 64 -> 128 -> 192 -> 256 -> 500
        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.Tanh(),

            nn.Linear(128, 192),
            nn.Tanh(),

            nn.Linear(192, 256),
            nn.Tanh(),

            nn.Linear(256, 500)
        )

    def forward(self, x):

        encoded = self.encoder(x)
        reconstructed = self.decoder(encoded)

        return reconstructed

In [8]:
# ============================================
# MODEL
# ============================================

model = Autoencoder()

print("\n============================================")
print("VANILLA AUTOENCODER")
print("============================================")

print(model)


VANILLA AUTOENCODER
Autoencoder(
  (encoder): Sequential(
    (0): Linear(in_features=500, out_features=256, bias=True)
    (1): Tanh()
    (2): Linear(in_features=256, out_features=192, bias=True)
    (3): Tanh()
    (4): Linear(in_features=192, out_features=128, bias=True)
    (5): Tanh()
    (6): Linear(in_features=128, out_features=64, bias=True)
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=128, bias=True)
    (1): Tanh()
    (2): Linear(in_features=128, out_features=192, bias=True)
    (3): Tanh()
    (4): Linear(in_features=192, out_features=256, bias=True)
    (5): Tanh()
    (6): Linear(in_features=256, out_features=500, bias=True)
  )
)


In [9]:
# ============================================
# LOSS AND OPTIMIZER
# ============================================

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

In [10]:
# ============================================
# TRAINING
# ============================================

print("\n============================================")
print("TRAINING")
print("============================================")

for epoch in range(EPOCHS):

    model.train()

    train_loss = 0.0

    for (batch,) in train_loader:

        optimizer.zero_grad()

        reconstructed = model(batch)

        loss = criterion(reconstructed, batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item() * batch.size(0)

    train_loss /= len(train_loader.dataset)

    # ----------------------------------------
    # DEVELOPMENT LOSS
    # ----------------------------------------

    model.eval()

    dev_loss = 0.0

    with torch.no_grad():

        for (batch,) in dev_loader:

            reconstructed = model(batch)

            loss = criterion(reconstructed, batch)

            dev_loss += loss.item() * batch.size(0)

    dev_loss /= len(dev_loader.dataset)

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"| Train Loss: {train_loss:.6f} "
        f"| Dev Loss: {dev_loss:.6f}"
    )


TRAINING
Epoch [1/30] | Train Loss: 0.010735 | Dev Loss: 0.010045
Epoch [2/30] | Train Loss: 0.009387 | Dev Loss: 0.009398
Epoch [3/30] | Train Loss: 0.008780 | Dev Loss: 0.009069
Epoch [4/30] | Train Loss: 0.008402 | Dev Loss: 0.008858
Epoch [5/30] | Train Loss: 0.008155 | Dev Loss: 0.008695
Epoch [6/30] | Train Loss: 0.007950 | Dev Loss: 0.008523
Epoch [7/30] | Train Loss: 0.007754 | Dev Loss: 0.008382
Epoch [8/30] | Train Loss: 0.007617 | Dev Loss: 0.008291
Epoch [9/30] | Train Loss: 0.007518 | Dev Loss: 0.008250
Epoch [10/30] | Train Loss: 0.007453 | Dev Loss: 0.008209
Epoch [11/30] | Train Loss: 0.007383 | Dev Loss: 0.008157
Epoch [12/30] | Train Loss: 0.007350 | Dev Loss: 0.008104
Epoch [13/30] | Train Loss: 0.007264 | Dev Loss: 0.008058
Epoch [14/30] | Train Loss: 0.007210 | Dev Loss: 0.008015
Epoch [15/30] | Train Loss: 0.007143 | Dev Loss: 0.007969
Epoch [16/30] | Train Loss: 0.007109 | Dev Loss: 0.007932
Epoch [17/30] | Train Loss: 0.007060 | Dev Loss: 0.007915
Epoch [18/30]

In [11]:
# ============================================
# TEST RECONSTRUCTION LOSS
# ============================================

model.eval()

test_loss = 0.0

with torch.no_grad():

    for (batch,) in test_loader:

        reconstructed = model(batch)

        loss = criterion(reconstructed, batch)

        test_loss += loss.item() * batch.size(0)

test_loss /= len(test_loader.dataset)

print("\n============================================")
print("FINAL RESULTS")
print("============================================")

print(f"Test reconstruction loss: {test_loss:.6f}")


FINAL RESULTS
Test reconstruction loss: 0.007942


In [12]:
# ============================================
# EXTRACT 64-D LATENT REPRESENTATIONS
# ============================================

with torch.no_grad():

    X_train_encoded = model.encoder(X_train_tensor).numpy()
    X_dev_encoded = model.encoder(X_dev_tensor).numpy()
    X_test_encoded = model.encoder(X_test_tensor).numpy()

In [13]:
# ============================================
# SAVE LATENT REPRESENTATIONS
# ============================================

np.save(
    "Data/processed/X_train_encoded.npy",
    X_train_encoded
)

np.save(
    "Data/processed/X_dev_encoded.npy",
    X_dev_encoded
)

np.save(
    "Data/processed/X_test_encoded.npy",
    X_test_encoded
)

print("\nLatent representations saved.")

print("Train latent:", X_train_encoded.shape)
print("Dev latent  :", X_dev_encoded.shape)
print("Test latent :", X_test_encoded.shape)


Latent representations saved.
Train latent: (5985, 64)
Dev latent  : (997, 64)
Test latent : (998, 64)
